In [1]:
import os
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1" # avoiding warnings
os.environ["HUGGINGFACE_HUB_VERBOSITY"] = "error"


import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from data import Preprocessor, create_dataloaders, GestureDataset, label_to_idx
from model import (LeNet5, create_pretrained_model, train_model,
                   make_mixup_fn, evaluate_with_tta, default_tta_transforms)


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## Data preprocessing & split

In [2]:
pre = Preprocessor(csv_path="../datasets/Train.csv", images_dir="../datasets/Images")
train_df, val_df, report = pre.split(return_report=True)

for key, value in report.items():
    print(f"{key}: {value}")

Removed 0 duplicate rows by img_IDS.
[WARNING] 1 row(s) have no matching image files — dropped.
rows_raw: 6249
removed_duplicate_ids: 0
rows_after_dedup: 6249
rows_with_files: 6248
rows_missing_files: 1
label_distribution_raw: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'Love': 694, 'You': 694, 'Friend': 694}
label_distribution_clean: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'You': 694, 'Friend': 694, 'Love': 693}


## LeNet-5

In [3]:
IMAGE_SIZE_LENET = (64, 64)
BATCH_SIZE = 32

base_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader, val_loader, label_to_idx = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_LENET,
    train_transform=base_transform,
    val_transform=base_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

NUM_CLASSES = len(label_to_idx)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Label mapping: {label_to_idx}")

Train batches: 130, Val batches: 65
Num classes: 9
Label mapping: {'Church': 0, 'Enough/Satisfied': 1, 'Friend': 2, 'Love': 3, 'Me': 4, 'Mosque': 5, 'Seat': 6, 'Temple': 7, 'You': 8}


## Train-val loop

In [ ]:
lenet = LeNet5(num_classes=NUM_CLASSES, dropout=0.3)

optimizer_l = torch.optim.Adam(lenet.parameters(), lr=3e-3)
scheduler_l = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_l, max_lr=3e-3, epochs=10,
    steps_per_epoch=len(train_loader),
)

result_lenet = train_model(
    model=lenet,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    optimizer=optimizer_l,
    scheduler=scheduler_l,
    scheduler_step_per_batch=True,
    device=device
)

best_auc = max(r["val_roc_auc"] for r in result_lenet["history"])
print(f"\nBest LeNet-5 val ROC AUC: {best_auc:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1939 | val_loss=2.1702 | val_roc_auc=0.6215
Epoch   2/10 | train_loss=2.0747 | val_loss=1.8840 | val_roc_auc=0.7572
Epoch   3/10 | train_loss=1.7519 | val_loss=1.6644 | val_roc_auc=0.8132
Epoch   4/10 | train_loss=1.5368 | val_loss=1.5178 | val_roc_auc=0.8529


## Model tuning

In [ ]:
IMAGE_SIZE_PRETRAINED = (224, 224)

pretrained_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader_pt, val_loader_pt, _ = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_PRETRAINED,
    train_transform=pretrained_transform,
    val_transform=pretrained_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

resnet18 = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_r = torch.optim.Adam(resnet18.parameters(), lr=1e-4)
scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_r, T_max=4)

result_resnet = train_model(
    model=resnet18,
    train_loader=train_loader_pt,
    val_loader=val_loader_pt,
    epochs=4,
    optimizer=optimizer_r,
    scheduler=scheduler_r,
    device=device
)

best_auc_r = max(r["val_roc_auc"] for r in result_resnet["history"])
print(f"\nBest ResNet18 val ROC AUC: {best_auc_r:.4f}")

Using device: mps
Epoch   1/4 | train_loss=2.1300 | val_loss=2.0105 | val_roc_auc=0.8281
Epoch   2/4 | train_loss=1.7447 | val_loss=1.3971 | val_roc_auc=0.9337
Epoch   3/4 | train_loss=1.2149 | val_loss=1.0561 | val_roc_auc=0.9584
Epoch   4/4 | train_loss=0.9813 | val_loss=0.9973 | val_roc_auc=0.9625

Best ResNet18 val ROC AUC: 0.9625


## Augmentations

In [ ]:
aug_train_transform = A.Compose([
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 16), hole_width_range=(8, 16), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader_aug, val_loader_aug, _ = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_PRETRAINED,
    train_transform=aug_train_transform,
    val_transform=pretrained_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

In [ ]:
resnet18_aug = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_a = torch.optim.Adam(resnet18_aug.parameters(), lr=1e-4)
scheduler_a = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_a, T_max=4)

result_aug = train_model(
    model=resnet18_aug,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=4,
    optimizer=optimizer_a,
    scheduler=scheduler_a,
    device=device
)

best_auc_aug = max(r["val_roc_auc"] for r in result_aug["history"])
print(f"\nBest ResNet18 + augmentations val ROC AUC: {best_auc_aug:.4f}")
print(f"Without augmentations: {best_auc_r:.4f}")
print(f"Gain: {best_auc_aug - best_auc_r:+.4f}")

Using device: mps
Epoch   1/4 | train_loss=2.1387 | val_loss=2.0227 | val_roc_auc=0.8064
Epoch   2/4 | train_loss=1.8253 | val_loss=1.4307 | val_roc_auc=0.9336
Epoch   3/4 | train_loss=1.3052 | val_loss=1.0366 | val_roc_auc=0.9595
Epoch   4/4 | train_loss=1.0569 | val_loss=0.9569 | val_roc_auc=0.9639

Best ResNet18 + augmentations val ROC AUC: 0.9639
Without augmentations: 0.9625
Gain: +0.0014


## MixUp & CutMix

In [ ]:
mixup_fn = make_mixup_fn(num_classes=NUM_CLASSES, alpha=0.4, mode="mixup")

resnet18_mix = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_m = torch.optim.Adam(resnet18_mix.parameters(), lr=1e-4)
scheduler_m = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_m, T_max=10)

result_mix = train_model(
    model=resnet18_mix,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=10,
    optimizer=optimizer_m,
    scheduler=scheduler_m,
    mixup_fn=mixup_fn,
    device=device
)

best_auc_mix = max(r["val_roc_auc"] for r in result_mix["history"])
print(f"\nResNet18 + aug + MixUp val ROC AUC: {best_auc_mix:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1700 | val_loss=2.1015 | val_roc_auc=0.7508
Epoch   2/10 | train_loss=2.0115 | val_loss=1.7391 | val_roc_auc=0.8959
Epoch   3/10 | train_loss=1.7483 | val_loss=1.3614 | val_roc_auc=0.9327
Epoch   4/10 | train_loss=1.5780 | val_loss=1.2752 | val_roc_auc=0.9395
Epoch   5/10 | train_loss=1.5525 | val_loss=1.2740 | val_roc_auc=0.9397
Epoch   6/10 | train_loss=1.4558 | val_loss=1.1825 | val_roc_auc=0.9466
Epoch   7/10 | train_loss=1.4007 | val_loss=0.9667 | val_roc_auc=0.9627
Epoch   8/10 | train_loss=1.1823 | val_loss=0.7302 | val_roc_auc=0.9775
Epoch   9/10 | train_loss=1.0741 | val_loss=0.5908 | val_roc_auc=0.9847
Epoch  10/10 | train_loss=1.0416 | val_loss=0.5174 | val_roc_auc=0.9877

ResNet18 + aug + MixUp val ROC AUC: 0.9877


In [ ]:
cutmix_fn = make_mixup_fn(num_classes=NUM_CLASSES, alpha=1.0, mode="cutmix")

resnet18_cut = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_c = torch.optim.Adam(resnet18_cut.parameters(), lr=1e-4)
scheduler_c = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=10)

result_cut = train_model(
    model=resnet18_cut,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=10,
    optimizer=optimizer_c,
    scheduler=scheduler_c,
    mixup_fn=cutmix_fn,
    device=device
)

best_auc_cut = max(r["val_roc_auc"] for r in result_cut["history"])
print(f"\nResNet18 + aug + CutMix val ROC AUC: {best_auc_cut:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1844 | val_loss=2.1293 | val_roc_auc=0.7125
Epoch   2/10 | train_loss=2.1208 | val_loss=1.9727 | val_roc_auc=0.8517
Epoch   3/10 | train_loss=2.0202 | val_loss=1.8195 | val_roc_auc=0.8970
Epoch   4/10 | train_loss=1.9361 | val_loss=1.7276 | val_roc_auc=0.9074
Epoch   5/10 | train_loss=1.9144 | val_loss=1.7644 | val_roc_auc=0.9062
Epoch   6/10 | train_loss=1.8956 | val_loss=1.6652 | val_roc_auc=0.9151
Epoch   7/10 | train_loss=1.8138 | val_loss=1.4592 | val_roc_auc=0.9363
Epoch   8/10 | train_loss=1.6489 | val_loss=1.0555 | val_roc_auc=0.9609
Epoch   9/10 | train_loss=1.5061 | val_loss=0.7843 | val_roc_auc=0.9766
Epoch  10/10 | train_loss=1.4250 | val_loss=0.6507 | val_roc_auc=0.9834

ResNet18 + aug + CutMix val ROC AUC: 0.9834


## Test-Time Augmentation

In [ ]:
all_labels = set(train_df["Label"].unique()) | set(val_df["Label"].unique())
lti = label_to_idx(all_labels)

val_ds_raw = GestureDataset(
    val_df,
    images_dir="../datasets/Images",
    label_to_idx=lti,
    image_size=IMAGE_SIZE_PRETRAINED,
    transform=None
)

best_model = result_mix["model"]

tta_transforms = default_tta_transforms(IMAGE_SIZE_PRETRAINED)

tta_result = evaluate_with_tta(
    best_model, val_ds_raw, tta_transforms, device
)

print(f"Standard ROC AUC (best model): {best_auc_mix:.4f}")
print(f"TTA ROC AUC:                   {tta_result['roc_auc']:.4f}")
print(f"Gain from TTA:                 {tta_result['roc_auc'] - best_auc_mix:+.4f}")

Standard ROC AUC (best model): 0.9625
TTA ROC AUC:                   0.9612
Gain from TTA:                 -0.0013


In [ ]:
print("=" * 55)
print(f"{'Experiment':<35} {'ROC AUC':>10}")
print("-" * 55)
print(f"{'LeNet-5 (baseline)':<35} {best_auc:>10.4f}")
print(f"{'ResNet18 (no aug)':<35} {best_auc_r:>10.4f}")
print(f"{'ResNet18 + augmentations':<35} {best_auc_aug:>10.4f}")
print(f"{'ResNet18 + aug + MixUp':<35} {best_auc_mix:>10.4f}")
print(f"{'ResNet18 + aug + CutMix':<35} {best_auc_cut:>10.4f}")
print(f"{'ResNet18 + aug + TTA':<35} {tta_result['roc_auc']:>10.4f}")
print("=" * 55)

Experiment                             ROC AUC
-------------------------------------------------------
LeNet-5 (baseline)                      0.8003
ResNet18 (no aug)                       0.9625
ResNet18 + augmentations                0.9639
ResNet18 + aug + MixUp                  0.9877
ResNet18 + aug + CutMix                 0.9834
ResNet18 + aug + TTA                    0.9612
